In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LangAnalysis").getOrCreate()
spark

In [3]:
posts = spark.read.format("xml") \
    .option("rowTag", "row") \
    .option("timestampFormat", "y-M-d H:m:s") \
    .load("posts_sample.xml")

posts.show(5)
print("Количество строк:", posts.count())

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|_Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+-

In [4]:
languages_df = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv("programming-languages.csv")

languages_df.show(5)

+----------+--------------------+
|      name|       wikipedia_url|
+----------+--------------------+
|   A# .NET|https://en.wikipe...|
|A# (Axiom)|https://en.wikipe...|
|A-0 System|https://en.wikipe...|
|        A+|https://en.wikipe...|
|       A++|https://en.wikipe...|
+----------+--------------------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import col, year

posts = posts.withColumn("Year", year(col("_CreationDate")))

posts = posts.filter((col("Year") >= 2010) & (col("Year") <= 2020))

posts.select("Year").show(5)

+----+
|Year|
+----+
|2010|
|2010|
|2010|
|2010|
|2010|
+----+
only showing top 5 rows


In [6]:
languages = [row[0].lower() for row in languages_df.collect()]
languages[:10]

['a# .net',
 'a# (axiom)',
 'a-0 system',
 'a+',
 'a++',
 'abap',
 'abc',
 'abc algol',
 'abset',
 'absys']

In [7]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def detect_language(tags):
    tags = str(tags).lower()
    for lang in languages:
        if f"<{lang}>" in tags:
            return lang
    return None

detect_udf = udf(detect_language, StringType())

In [8]:
df = posts.withColumn("Language", detect_udf(col("_Tags")))

df = df.dropna(subset=["Language"])

df.select("Year", "Language").show(5)

+----+--------+
|Year|Language|
+----+--------+
|2010|    java|
|2010|     php|
|2010|    ruby|
|2010|       c|
|2010|     php|
+----+--------+
only showing top 5 rows


In [9]:
result = df.groupBy("Year", "Language").count()

result.show(10)

+----+-----------+-----+
|Year|   Language|count|
+----+-----------+-----+
|2019| typescript|    6|
|2017|       perl|    6|
|2011|    haskell|    1|
|2012|       bash|    9|
|2018|       glsl|    1|
|2011|objective-c|   33|
|2013| processing|    1|
|2013|        php|  173|
|2013|       chef|    2|
|2017|       curl|    5|
+----+-----------+-----+
only showing top 10 rows


In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("Year").orderBy(col("count").desc())

top10 = result.withColumn("rank", row_number().over(window)) \
              .filter(col("rank") <= 10)

top10.show(50)

+----+-----------+-----+----+
|Year|   Language|count|rank|
+----+-----------+-----+----+
|2010|       java|   52|   1|
|2010| javascript|   44|   2|
|2010|        php|   42|   3|
|2010|     python|   25|   4|
|2010|objective-c|   23|   5|
|2010|          c|   20|   6|
|2010|       ruby|   11|   7|
|2010|     delphi|    7|   8|
|2010|applescript|    3|   9|
|2010|          r|    3|  10|
|2011|        php|   97|   1|
|2011|       java|   92|   2|
|2011| javascript|   82|   3|
|2011|     python|   35|   4|
|2011|objective-c|   33|   5|
|2011|          c|   24|   6|
|2011|       ruby|   17|   7|
|2011|     delphi|    8|   8|
|2011|       perl|    8|   9|
|2011|       bash|    7|  10|
|2012|        php|  136|   1|
|2012| javascript|  129|   2|
|2012|       java|  124|   3|
|2012|     python|   65|   4|
|2012|objective-c|   45|   5|
|2012|          c|   27|   6|
|2012|       ruby|   25|   7|
|2012|       bash|    9|   8|
|2012|          r|    9|   9|
|2012|      scala|    6|  10|
|2013| jav

In [11]:
top10.write.mode("overwrite").parquet("top_ten_languages.parquet")

In [12]:
check = spark.read.parquet("top_ten_languages.parquet")
check.show(50)

+----+-----------+-----+----+
|Year|   Language|count|rank|
+----+-----------+-----+----+
|2010|       java|   52|   1|
|2010| javascript|   44|   2|
|2010|        php|   42|   3|
|2010|     python|   25|   4|
|2010|objective-c|   23|   5|
|2010|          c|   20|   6|
|2010|       ruby|   11|   7|
|2010|     delphi|    7|   8|
|2010|applescript|    3|   9|
|2010|          r|    3|  10|
|2011|        php|   97|   1|
|2011|       java|   92|   2|
|2011| javascript|   82|   3|
|2011|     python|   35|   4|
|2011|objective-c|   33|   5|
|2011|          c|   24|   6|
|2011|       ruby|   17|   7|
|2011|     delphi|    8|   8|
|2011|       perl|    8|   9|
|2011|       bash|    7|  10|
|2012|        php|  136|   1|
|2012| javascript|  129|   2|
|2012|       java|  124|   3|
|2012|     python|   65|   4|
|2012|objective-c|   45|   5|
|2012|          c|   27|   6|
|2012|       ruby|   25|   7|
|2012|       bash|    9|   8|
|2012|          r|    9|   9|
|2012|      scala|    6|  10|
|2013| jav